# How many significant eQTLs (SNP–gene pairs) did we find?

Run the cells in order. Set `ONLY_22 = True` to analyze chromosome 22, or `False` to analyze chromosomes 1–22. The European–Yoruba comparison uses chromosome 22 in either case.

The input data is in `checkpoint_data/`.

## Functions used in the analysis

In [ ]:
"""Functions for eQTL tests and population comparisons, adjusted for genotype PCs."""
import numpy as np
from scipy import stats


def _empty_associations():
    return tuple(np.array([], dtype=int if i < 2 else float) for i in range(5))


def _combine_associations(pieces):
    if not pieces:
        return _empty_associations()
    values = tuple(np.concatenate([piece[i] for piece in pieces]) for i in range(5))
    order = np.argsort(values[0])
    return tuple(v[order] for v in values)


def _ols_statistics(genotype_sum_squares, genotype_expression_product, expression_sum_squares, degrees_freedom):
    beta = genotype_expression_product / genotype_sum_squares
    standard_error = np.sqrt(np.maximum(expression_sum_squares - genotype_expression_product * beta,
        0) / degrees_freedom / genotype_sum_squares)
    with np.errstate(divide="ignore", invalid="ignore"):
        p_value = 2 * stats.t.sf(np.abs(beta / standard_error), degrees_freedom)
    valid = np.isfinite(beta) & np.isfinite(standard_error) & np.isfinite(p_value)
    return beta, standard_error, p_value, valid


def covariate_basis(covariates):
    covariate_matrix = np.asarray(covariates, dtype=float)
    if not np.isfinite(covariate_matrix).all() or np.linalg.matrix_rank(covariate_matrix) != covariate_matrix.shape[1]:
        raise ValueError("Covariates must be finite and have independent columns.")
    return np.linalg.qr(covariate_matrix, mode="reduced")[0]


def associate_adjusted(expression_values, genotypes, indices, covariates, cache=None):
    """Test SNP associations with linear regression, adjusting for covariates.

    Use the available samples for each SNP–gene pair."""
    indices = np.asarray(indices, dtype=int)
    if not len(indices):
        return _empty_associations()
    if cache is not None and np.isfinite(expression_values).all():
        covariate_q = cache["Q"]
        adjusted_expression = expression_values - covariate_q @ (covariate_q.T @ expression_values)
        expression_sum_squares = adjusted_expression @ adjusted_expression
        if expression_sum_squares <= 1e-12:
            return _empty_associations()
        complete = cache["complete"][indices]
        good_indices = indices[complete & (cache["xx"][indices] > 1e-12)]
        pieces = []
        if len(good_indices):
            genotype_sum_squares = cache["xx"][good_indices]
            genotype_expression_product = cache["R"][:, good_indices].T @ adjusted_expression
            degrees_freedom = len(expression_values) - covariates.shape[1] - 1
            beta, standard_error, p_value, valid = _ols_statistics(genotype_sum_squares,
                genotype_expression_product,
                expression_sum_squares,
                degrees_freedom)
            pieces.append((good_indices[valid],
                np.full(valid.sum(),
                len(expression_values),
                dtype=int),
                beta[valid],
                standard_error[valid],
                p_value[valid]))
        if (~complete).any():
            pieces.append(associate_adjusted(expression_values, genotypes, indices[~complete], covariates))
        return _combine_associations(pieces)
    x = genotypes[:, indices]
    ok = np.isfinite(x) & np.isfinite(expression_values[:, None]) & np.isfinite(covariates).all(axis=1)[:, None]
    if ok.all():
        groups = [(np.ones(len(expression_values), dtype=bool), np.arange(len(indices)))]
    else:
        _, labels = np.unique(np.packbits(ok, axis=0).T, axis=0, return_inverse=True)
        groups = [(ok[:, np.flatnonzero(labels == g)[0]], np.flatnonzero(labels == g))
                  for g in np.unique(labels)]
    pieces = []
    for rows, columns in groups:
        covariate_matrix = covariates[rows]
        sample_count = int(rows.sum())
        if sample_count <= covariate_matrix.shape[1] + 1 or np.linalg.matrix_rank(covariate_matrix) != covariate_matrix.shape[1]:
            continue
        covariate_q = covariate_basis(covariate_matrix)
        adjusted_expression = expression_values[rows] - covariate_q @ (covariate_q.T @ expression_values[rows])
        expression_sum_squares = adjusted_expression @ adjusted_expression
        if expression_sum_squares <= 1e-12:
            continue
        adjusted_dosages = x[np.ix_(rows, columns)]
        adjusted_dosages = adjusted_dosages - covariate_q @ (covariate_q.T @ adjusted_dosages)
        genotype_sum_squares = np.einsum("ij,ij->j", adjusted_dosages, adjusted_dosages)
        valid = genotype_sum_squares > 1e-12
        if not valid.any():
            continue
        tested_indices = indices[columns[valid]]
        genotype_expression_product = adjusted_dosages[:, valid].T @ adjusted_expression
        degrees_freedom = sample_count - covariate_matrix.shape[1] - 1
        beta, standard_error, p_value, finite = _ols_statistics(genotype_sum_squares[valid],
            genotype_expression_product,
            expression_sum_squares,
            degrees_freedom)
        pieces.append((tested_indices[finite],
            np.full(finite.sum(),
            sample_count,
            dtype=int),
            beta[finite],
            standard_error[finite],
            p_value[finite]))
    return _combine_associations(pieces)


def check_adjusted_pair(expression_values, dosage, covariates, beta, standard_error, p_value):
    complete = np.isfinite(expression_values) & np.isfinite(dosage) & np.isfinite(covariates).all(axis=1)
    design_matrix = np.column_stack([covariates[complete], dosage[complete]])
    complete_expression = expression_values[complete]
    coefficients = np.linalg.lstsq(design_matrix, complete_expression, rcond=None)[0]
    degrees_freedom = len(complete_expression) - design_matrix.shape[1]
    direct_se = np.sqrt(np.sum((complete_expression - design_matrix @ coefficients)**2) / degrees_freedom * np.linalg.inv(design_matrix.T @ design_matrix)[-1,
        -1])
    direct_p = 2 * stats.t.sf(abs(coefficients[-1] / direct_se), degrees_freedom)
    np.testing.assert_allclose([beta,
        standard_error,
        p_value],
        [coefficients[-1],
        direct_se,
        direct_p],
        rtol=1e-6,
        atol=1e-10)


def relationship_context(genotypes, positions, marker_mask):
    """Build a genetic relationship matrix from standardized, LD-pruned SNPs."""
    standardized_genotypes = genotypes[:, marker_mask].copy()
    allele_frequency = standardized_genotypes.mean(axis=0) / 2
    if not np.isfinite(standardized_genotypes).all() or not ((allele_frequency > 0) & (allele_frequency < 1)).all():
        raise ValueError("PCA markers must be complete and variable.")
    standardized_genotypes -= 2 * allele_frequency
    standardized_genotypes /= np.sqrt(2 * allele_frequency * (1 - allele_frequency))
    return {"Z": standardized_genotypes,
        "positions": np.asarray(positions)[marker_mask],
        "K": standardized_genotypes @ standardized_genotypes.T}


def population_covariates(context, start, end, is_yri, n_pcs=5):
    """Calculate PCs for both populations, leaving out this gene's test window."""
    inside = (context["positions"] >= start) & (context["positions"] <= end)
    remaining = context["Z"].shape[1] - int(inside.sum())
    if remaining < 100:
        raise ValueError("Too few PCA markers outside the cis window.")
    removed = context["Z"][:, inside]
    K = (context["K"] - removed @ removed.T) / remaining
    values, vectors = np.linalg.eigh(K)
    pcs = vectors[:, np.argsort(values)[::-1][:n_pcs]].copy()
    # Subtract each population’s mean from its PCs.
    for group in (False, True):
        rows = np.asarray(is_yri, dtype=bool) == group
        pcs[rows] -= pcs[rows].mean(axis=0)
    scale = pcs.std(axis=0)
    if (scale < 1e-10).any():
        raise ValueError("A genotype PC is redundant with population membership.")
    pcs /= scale
    covariate_matrix = np.column_stack([np.ones(len(is_yri)), np.asarray(is_yri, float), pcs])
    return covariate_matrix, remaining


def interaction_hc3(expression_values, genotypes, is_yri, covariates):
    """Test whether SNP effects differ between populations, adjusting for genotype PCs.

    Return the EUR and YRI effects, their difference, its HC3 standard error,
    the p-value, and a mask of SNPs that could be tested. P-values use a
    normal approximation.
    """
    covariate_q = covariate_basis(covariates)
    adjusted_expression = expression_values - covariate_q @ (covariate_q.T @ expression_values)
    adjusted_snps = genotypes - covariate_q @ (covariate_q.T @ genotypes)
    interaction = genotypes * np.asarray(is_yri, float)[:, None]
    adjusted_interactions = interaction - covariate_q @ (covariate_q.T @ interaction)
    snp_sum_squares = np.einsum("ij,ij->j", adjusted_snps, adjusted_snps)
    snp_interaction_product = np.einsum("ij,ij->j", adjusted_snps, adjusted_interactions)
    interaction_sum_squares = np.einsum("ij,ij->j", adjusted_interactions, adjusted_interactions)
    determinant = snp_sum_squares * interaction_sum_squares - snp_interaction_product * snp_interaction_product
    estimable = (snp_sum_squares > 1e-10) & (interaction_sum_squares > 1e-10) & (determinant > 1e-10 * np.maximum(snp_sum_squares * interaction_sum_squares,
        1))
    safe_determinant = np.where(estimable, determinant, 1)
    snp_expression_product, interaction_expression_product = adjusted_snps.T @ adjusted_expression, adjusted_interactions.T @ adjusted_expression
    beta_eur = (interaction_sum_squares * snp_expression_product - snp_interaction_product * interaction_expression_product) / safe_determinant
    difference = (snp_sum_squares * interaction_expression_product - snp_interaction_product * snp_expression_product) / safe_determinant
    residual = adjusted_expression[:, None] - adjusted_snps * beta_eur - adjusted_interactions * difference
    covariate_leverage = np.einsum("ij,ij->i", covariate_q, covariate_q)[:, None]
    leverage = covariate_leverage + (interaction_sum_squares * adjusted_snps * adjusted_snps - 2 * snp_interaction_product * adjusted_snps * adjusted_interactions + snp_sum_squares * adjusted_interactions * adjusted_interactions) / safe_determinant
    influence = (-snp_interaction_product * adjusted_snps + snp_sum_squares * adjusted_interactions) / safe_determinant
    with np.errstate(divide="ignore", invalid="ignore"):
        variance = ((influence * residual / (1 - leverage))**2).sum(axis=0)
        standard_error = np.sqrt(variance)
        p_value = 2 * stats.norm.sf(np.abs(difference / standard_error))
    valid = estimable & np.isfinite(p_value) & np.isfinite(standard_error) & (standard_error > 0) & (leverage < 1 - 1e-10).all(axis=0)
    return beta_eur[valid], (beta_eur + difference)[valid], difference[valid], standard_error[valid], p_value[valid], valid


def check_interaction(expression_values, dosage, is_yri, covariates, difference, standard_error, p_value):
    design_matrix = np.column_stack([covariates, dosage, dosage * np.asarray(is_yri, float)])
    inverse = np.linalg.inv(design_matrix.T @ design_matrix)
    coefficients = np.linalg.lstsq(design_matrix, expression_values, rcond=None)[0]
    residual = expression_values - design_matrix @ coefficients
    leverage = np.einsum("ij,jk,ik->i", design_matrix, inverse, design_matrix)
    weighted = design_matrix * (residual / (1 - leverage))[:, None]
    covariance = inverse @ (weighted.T @ weighted) @ inverse
    direct_se = np.sqrt(covariance[-1, -1])
    direct_p = 2 * stats.norm.sf(abs(coefficients[-1] / direct_se))
    np.testing.assert_allclose([difference,
        standard_error,
        p_value],
        [coefficients[-1],
        direct_se,
        direct_p],
        rtol=1e-5,
        atol=1e-9)


def genotype_adjustment_cache(genotypes, covariates):
    """Save adjusted genotypes for SNPs with no missing values to avoid repeating calculations."""
    covariate_q = covariate_basis(covariates)
    complete = np.isfinite(genotypes).all(axis=0)
    adjusted_genotypes = np.zeros_like(genotypes, dtype=float)
    adjusted_genotypes[:,
        complete] = genotypes[:,
        complete] - covariate_q @ (covariate_q.T @ genotypes[:,
        complete])
    return {"Q": covariate_q,
        "complete": complete,
        "R": adjusted_genotypes,
        "xx": np.einsum("ij,ij->j",
        adjusted_genotypes,
        adjusted_genotypes)}


In [ ]:
ONLY_22 = False

from pathlib import Path
import os
import gzip
import json
import re
import subprocess
import time
import numpy as np
import pandas as pd
from IPython.display import display

DATA = Path("checkpoint_data")
GENOTYPES = DATA / "LDREF"
CHROMOSOMES = (22,) if ONLY_22 else tuple(range(1, 23))
N_PCS = 5
CIS_FLANK_BP = 500_000
ALPHA = 0.05
OUT = Path("results") / ("chr22_pca" if ONLY_22 else "project_checkpoint_pca")
WORK = OUT / "work"
WORK.mkdir(parents=True, exist_ok=True)
PLINK = Path("tools") / ("plink2.exe" if os.name == "nt" else "plink2")
EXPRESSION = DATA / "expression" / ("GEUVADIS_chr22.tsv.gz" if ONLY_22 else "GEUVADIS_full.tsv.gz")
GTF = DATA / "annotations/gencode.v12.annotation.gtf.gz"
ANNOT22 = [DATA / "annotations/chr22_GRCh37_genes.json",
           DATA / "annotations/chr22_historical_recovered_genes.json"]
PCA_DIR = DATA / "covariates/eur_loco"
BIM_COLUMNS = ["chrom", "snp", "cm", "position", "a1", "a2"]
(OUT / "commands.jsonl").write_text("")

def run_plink(arguments):
    command = [str(PLINK), *map(str, arguments)]
    result = subprocess.run(command, capture_output=True, text=True)
    with (OUT / "commands.jsonl").open("a") as handle:
        handle.write(json.dumps({"command": command, "returncode": result.returncode}) + "\n")
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)

print(subprocess.check_output([str(PLINK), "--version"], text=True).strip())
print("Output folder:", OUT)


PLINK v2.0.0-b.1-dev M1 (12 Oct 2026)
Output folder: results/chr22_pca


## Matching the data

We match the genotype and expression samples by ID. For each gene, we test SNPs inside the gene and up to 500 kb on either side, using GRCh37 coordinates.

In [2]:
expression = pd.read_csv(EXPRESSION, sep="\t")
expression["gene_id"] = expression["TargetID"].str.split(".").str[0]
assert expression.gene_id.is_unique
fam_columns = ["FID", "IID", "father", "mother", "sex", "phenotype"]
fam = pd.read_csv(GENOTYPES / f"1000G.EUR.{CHROMOSOMES[0]}.fam", sep=r"\s+", header=None,
                  names=fam_columns, dtype=str)
assert fam.IID.is_unique
for chrom in CHROMOSOMES[1:]:
    other = pd.read_csv(GENOTYPES / f"1000G.EUR.{chrom}.fam", sep=r"\s+",
                        header=None, names=fam_columns, dtype=str)
    assert set(zip(other.FID, other.IID)) == set(zip(fam.FID, fam.IID))
matched = fam.loc[fam.IID.isin(expression.columns), ["FID", "IID"]].copy()
assert len(matched) > 2
matched.to_csv(OUT / "matched_samples.tsv", sep="\t", index=False)
KEEP = WORK / "keep.txt"
matched.to_csv(KEEP, sep="\t", index=False, header=False)
SAMPLE_IDS = matched.IID.tolist()

attributes = re.compile(r'(\w+) "([^"]*)"')
records = []
if any(c != 22 for c in CHROMOSOMES):
    with gzip.open(GTF, "rt") as handle:
        for line in handle:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) != 9 or f[2] != "gene":
                continue
            chrom = f[0].removeprefix("chr")
            if not chrom.isdigit() or int(chrom) not in CHROMOSOMES or int(chrom) == 22:
                continue
            a = dict(attributes.findall(f[8]))
            records.append({"gene_id": a["gene_id"].split(".")[0], "chrom": int(chrom),
                            "start": int(f[3]), "end": int(f[4]), "symbol": a.get("gene_name", ""),
                            "annotation_source": "GENCODE v12 GRCh37"})
for path in (ANNOT22 if 22 in CHROMOSOMES else []):
    for g in json.loads(path.read_text()):
        assert g["assembly_name"] == "GRCh37"
        if str(g["seq_region_name"]) == "22":
            records.append({"gene_id": g["id"], "chrom": 22, "start": int(g["start"]),
                            "end": int(g["end"]), "symbol": g.get("external_name", g["id"]),
                            "annotation_source": "saved Week 4 GRCh37 records"})
annotation = pd.DataFrame(records)
assert annotation.gene_id.is_unique and (annotation.end >= annotation.start).all()
genes = annotation.merge(expression[["gene_id", "TargetID", "Chr", "Coord"]], on="gene_id", validate="one_to_one")
assert genes.chrom.astype(str).eq(genes.Chr.astype(str)).all()
genes["expression_coord_within_gene"] = genes.Coord.between(genes.start, genes.end)
genes.loc[~genes.expression_coord_within_gene].to_csv(OUT / "annotation_coordinate_review.tsv", sep="\t", index=False)
assert genes.loc[genes.chrom.ne(22), "expression_coord_within_gene"].all()
genes["window_start"] = (genes.start - CIS_FLANK_BP).clip(lower=1)
genes["window_end"] = genes.end + CIS_FLANK_BP
genes = genes.sort_values(["chrom", "start", "gene_id"]).reset_index(drop=True)
genes.to_csv(OUT / "analyzed_gene_annotations.tsv", sep="\t", index=False)
expression_by_gene = expression.set_index("gene_id")
Y_TABLE = expression_by_gene.loc[:, SAMPLE_IDS].astype(float)
missing = expression.loc[~expression.gene_id.isin(annotation.gene_id), ["TargetID", "gene_id", "Chr", "Coord"]]
missing.to_csv(OUT / "expression_without_autosomal_annotation.tsv", sep="\t", index=False)
print(f"Genotype samples: {len(fam)}; expression samples: {len(expression.columns)-5}; matched: {len(matched)}")
print(f"Measured annotated autosomal genes: {len(genes):,}")



Genotype samples: 489; expression samples: 462; matched: 344
Measured annotated autosomal genes: 633


## Testing SNP–gene associations

We use linear regression to test whether each SNP is associated with gene expression, adjusting for five genotype PCs. These PCs are calculated without the chromosome being tested.

Model: `expression ~ SNP + five PCs`.

In [3]:
PAIR_FILE = WORK / "all_cis_pairs.tsv.gz"
columns = ["chrom", "gene_id", "symbol", "snp", "position", "counted_allele", "other_allele", "n", "beta", "se", "p_value"]
gene_summaries = []
total_tests = checked_pairs = 0

with gzip.open(PAIR_FILE, "wt") as handle:
    handle.write("\t".join(columns) + "\n")
    for chrom in CHROMOSOMES:
        started = time.monotonic()
        original = GENOTYPES / f"1000G.EUR.{chrom}"
        prefix = WORK / f"full_chr{chrom}"
        run_plink(["--bfile", original, "--keep", KEEP, "--snps-only", "just-acgt", "--export", "A", "--out", prefix])
        raw = pd.read_csv(str(prefix) + ".raw", sep=r"\s+", dtype={"FID": str, "IID": str})
        assert raw.IID.is_unique and set(raw.IID) == set(SAMPLE_IDS)
        raw = raw.set_index("IID").loc[SAMPLE_IDS]
        assert raw.FID.tolist() == matched.FID.tolist()
        bim = pd.read_csv(str(original) + ".bim", sep=r"\s+", header=None,
                          names=BIM_COLUMNS, dtype={"snp": str})
        assert bim.snp.is_unique and bim.chrom.eq(chrom).all()
        dosage_columns = list(raw.columns[5:])
        split = [c.rsplit("_", 1) for c in dosage_columns]
        assert [x[0] for x in split] == bim.snp.tolist(), "BIM/export SNP order mismatch."
        bim["counted_allele"] = [x[1] for x in split]
        X = raw.loc[:, dosage_columns].to_numpy(dtype=float)
        assert ((X[np.isfinite(X)] >= 0) & (X[np.isfinite(X)] <= 2)).all()
        order = np.argsort(bim.position.to_numpy(), kind="stable")
        bim = bim.iloc[order].reset_index(drop=True)
        X = X[:, order]
        positions = bim.position.to_numpy()
        pc_table = pd.read_csv(PCA_DIR / f"eur_loco_chr{chrom}.tsv", sep="\t", dtype={"FID": str, "IID": str})
        assert pc_table.IID.is_unique
        pc_table = pc_table.set_index("IID").loc[SAMPLE_IDS]
        assert pc_table.FID.tolist() == matched.FID.tolist()
        covariates = np.column_stack([np.ones(len(matched)), pc_table[[f"PC{i+1}" for i in range(N_PCS)]].to_numpy()])
        assert np.isfinite(covariates).all() and np.linalg.matrix_rank(covariates) == N_PCS+1
        regression_cache = genotype_adjustment_cache(X, covariates)

        chrom_tests = 0
        del raw
        for gene in genes.loc[genes.chrom.eq(chrom)].itertuples(index=False):
            y = Y_TABLE.loc[gene.gene_id].to_numpy(dtype=float)
            left = np.searchsorted(positions, gene.window_start, side="left")
            right = np.searchsorted(positions, gene.window_end, side="right")
            ix, n, beta, se, pvals = associate_adjusted(
                y, X, np.arange(left, right), covariates, regression_cache)
            base = {"chrom": chrom, "gene_id": gene.gene_id, "symbol": gene.symbol,
                    "window_start": gene.window_start, "window_end": gene.window_end}
            if not len(ix):
                gene_summaries.append({**base, "n_tests": 0, "status": "no variable complete-case SNP-expression pair", "lead_snp": "", "lead_p": np.nan})
                continue
            if chrom_tests == 0:
                k = int(np.argmin(np.abs(pvals - 0.1)))
                j = ix[k]
                check_adjusted_pair(y, X[:, j], covariates, beta[k], se[k], pvals[k])
                checked_pairs += 1
            v = bim.iloc[ix]
            other = np.where(v.counted_allele.eq(v.a1), v.a2, v.a1)
            result = pd.DataFrame({"chrom": chrom, "gene_id": gene.gene_id, "symbol": gene.symbol,
                "snp": v.snp.to_numpy(), "position": v.position.to_numpy(),
                "counted_allele": v.counted_allele.to_numpy(), "other_allele": other,
                "n": n, "beta": beta, "se": se, "p_value": pvals})
            result.to_csv(handle, sep="\t", index=False, header=False)
            k = int(np.argmin(pvals))
            gene_summaries.append({**base, "n_tests": len(result), "status": "tested",
                                   "lead_snp": result.snp.iloc[k], "lead_p": pvals[k]})
            chrom_tests += len(result)
            total_tests += len(result)
        print(f"Chr {chrom:2}: {chrom_tests:,} valid SNP–gene tests ({time.monotonic()-started:.1f}s)")
        del X, regression_cache
summary = pd.DataFrame(gene_summaries)
assert summary.gene_id.is_unique and int(summary.n_tests.sum()) == total_tests and total_tests > 0
print(f"Total: {total_tests:,} tests; {checked_pairs} direct regression checks passed.")


Chr 22: 280,105 valid SNP–gene tests (3.8s)
Total: 280,105 tests; 1 direct regression checks passed.


## Counting significant pairs

We use Bonferroni correction to account for the number of tests. The cutoff is `0.05 / number of tests`. We calculate it separately for each chromosome and for all selected chromosomes together.

In [4]:
by_chrom = summary.groupby("chrom").agg(genes_considered=("gene_id", "size"), tested_pairs=("n_tests", "sum")).reindex(CHROMOSOMES, fill_value=0)
assert by_chrom.tested_pairs.gt(0).all()
by_chrom["p_threshold_chromosome"] = ALPHA / by_chrom.tested_pairs
scan_threshold = ALPHA / total_tests
by_chrom["p_threshold_scan"] = scan_threshold
chrom_significant = {c: 0 for c in CHROMOSOMES}
scan_significant = {c: 0 for c in CHROMOSOMES}
rows_read = 0
CHROM_FILE = OUT / "significant_pairs_chromosome_bonferroni.tsv.gz"
SCAN_FILE = OUT / "significant_pairs_scan_bonferroni.tsv.gz"
with gzip.open(CHROM_FILE, "wt") as ch, gzip.open(SCAN_FILE, "wt") as sc:
    header = "\t".join(columns + ["p_bonferroni_chromosome", "p_bonferroni_scan"]) + "\n"
    ch.write(header)
    sc.write(header)
    for chunk in pd.read_csv(PAIR_FILE, sep="\t", chunksize=100_000):
        assert chunk.p_value.between(0, 1).all()
        rows_read += len(chunk)
        m = chunk.chrom.map(by_chrom.tested_pairs)
        chunk["p_bonferroni_chromosome"] = np.minimum(chunk.p_value * m, 1)
        chunk["p_bonferroni_scan"] = np.minimum(chunk.p_value * total_tests, 1)
        sig_ch = chunk.loc[chunk.p_value.le(ALPHA / m)]
        sig_sc = chunk.loc[chunk.p_value.le(scan_threshold)]
        sig_ch.to_csv(ch, sep="\t", index=False, header=False)
        sig_sc.to_csv(sc, sep="\t", index=False, header=False)
        for c, count in sig_ch.groupby("chrom").size().items():
            chrom_significant[int(c)] += int(count)
        for c, count in sig_sc.groupby("chrom").size().items():
            scan_significant[int(c)] += int(count)
assert rows_read == total_tests
by_chrom["significant_pairs_chromosome"] = [chrom_significant[c] for c in CHROMOSOMES]
by_chrom["significant_pairs_scan"] = [scan_significant[c] for c in CHROMOSOMES]
by_chrom.to_csv(OUT / "counts_by_chromosome.tsv", sep="\t")
chrom_total = sum(chrom_significant.values())
scan_total = sum(scan_significant.values())
summary["lead_p_bonferroni_chromosome"] = np.minimum(summary.lead_p * summary.chrom.map(by_chrom.tested_pairs), 1)
summary["lead_p_bonferroni_scan"] = np.minimum(summary.lead_p * total_tests, 1)
summary.to_csv(OUT / "gene_test_summary.tsv", sep="\t", index=False)
print(f"Model: SNP + {N_PCS} genotype PCs (leave tested chromosome out).")
print(f"Selected chromosomes: {list(CHROMOSOMES)}; {total_tests:,} SNP–gene tests.")
print(f"Chromosome-wide correction: {chrom_total:,} significant pairs (sum of per-chromosome counts).")
print(f"Scan-wide correction: {scan_total:,} significant pairs; raw-p threshold {scan_threshold:.6g}.")
display(by_chrom)

settings = {"chromosomes": list(CHROMOSOMES), "samples": len(matched),
            "model": "OLS adjusted for genome-wide LOCO genotype PCs", "n_pcs": N_PCS, "cis_flank_bp": CIS_FLANK_BP,
            "tests": total_tests, "chromosome_corrected_pairs": chrom_total,
            "scan_corrected_pairs": scan_total, "scan_p_threshold": scan_threshold,
            "alpha": ALPHA}
(OUT / "run_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print("Saved results to:", OUT)


Model: SNP + 5 genotype PCs (leave tested chromosome out).
Selected chromosomes: [22]; 280,105 SNP–gene tests.
Chromosome-wide correction: 1,302 significant pairs (sum of per-chromosome counts).
Scan-wide correction: 1,302 significant pairs; raw-p threshold 1.78504e-07.
Saved results to: results/chr22_pca


,genes_considered,tested_pairs,p_threshold_chromosome,p_threshold_scan,significant_pairs_chromosome,significant_pairs_scan
chrom,,,,,,
22,633,280105,1.785045e-07,1.785045e-07,1302,1302


# How many eGenes did we find?

An eGene is a gene with at least one significant SNP association.

In [5]:
gene_results = summary.copy()
tested = gene_results.n_tests.gt(0)
gene_results["eGene_chromosome"] = tested & gene_results.lead_p.le(
    gene_results.chrom.map(by_chrom.p_threshold_chromosome))
gene_results["eGene_scan"] = tested & gene_results.lead_p.le(scan_threshold)
egenes = gene_results.assign(tested=tested).groupby("chrom").agg(
    genes_tested=("tested", "sum"),
    eGenes_chromosome=("eGene_chromosome", "sum"),
    eGenes_scan=("eGene_scan", "sum"))
gene_results.loc[gene_results.eGene_chromosome].to_csv(
    OUT / "egenes_chromosome_bonferroni.tsv", sep="\t", index=False)
gene_results.loc[gene_results.eGene_scan].to_csv(
    OUT / "egenes_scan_bonferroni.tsv", sep="\t", index=False)
egenes.to_csv(OUT / "egenes_by_chromosome.tsv", sep="\t")
assert set(egenes.index) == set(CHROMOSOMES)
print(f"{int(egenes.genes_tested.sum()):,} genes tested")
print(f"Chromosome-wide correction: {int(egenes.eGenes_chromosome.sum()):,} eGenes")
print(f"Scan-wide correction: {int(egenes.eGenes_scan.sum()):,} eGenes")
display(egenes)


633 genes tested
Chromosome-wide correction: 74 eGenes
Scan-wide correction: 74 eGenes


,genes_tested,eGenes_chromosome,eGenes_scan
chrom,,,
22,633,74,74


# Do SNP effects differ between European and Yoruba samples?

For chromosome 22, we compare 358 European (EUR) and 87 Yoruba (YRI) samples. We add a SNP-by-population interaction to test whether the SNP’s association with expression differs between the two groups.

Model: `expression ~ population + SNP + SNP:population + five PCs`.

We use HC3 standard errors and Bonferroni correction. The PCs come from chromosome 22 markers outside each gene’s test window, so they may not fully account for population structure.

In [6]:
POP_DATA = DATA / "populations/chr22_EUR_YRI"
POP_OUT = Path("results/population_chr22_pca")
POP_PREFIX = POP_DATA / "chr22_EUR_YRI"
pop_raw_prefix = POP_OUT / "work/dosages"
pop_raw_prefix.parent.mkdir(parents=True, exist_ok=True)
run_plink(["--bfile", POP_PREFIX, "--export", "A", "--out", pop_raw_prefix])
pop_raw = pd.read_csv(str(pop_raw_prefix) + ".raw", sep=r"\s+", dtype={"FID": str, "IID": str})
pop_meta = pd.read_csv(POP_DATA / "sample_populations.tsv", sep="\t", dtype=str)
assert pop_raw.IID.is_unique and pop_meta["sample"].is_unique
pop_labels = pop_meta.set_index("sample").loc[pop_raw.IID, "analysis_population"].to_numpy()
assert set(pop_labels) == {"EUR", "YRI"}
pop_masks = {group: pop_labels == group for group in ("EUR", "YRI")}
assert {k: int(v.sum()) for k, v in pop_masks.items()} == {"EUR": 358, "YRI": 87}
pop_variants = pd.read_csv(str(POP_PREFIX) + ".bim", sep=r"\s+", header=None,
    names=BIM_COLUMNS)
assert pop_variants.snp.is_unique and pop_variants.chrom.eq(22).all()
pop_columns = list(pop_raw.columns[6:])
pop_split = [c.rsplit("_", 1) for c in pop_columns]
assert [v[0] for v in pop_split] == pop_variants.snp.tolist()
pop_variants["counted_allele"] = [v[1] for v in pop_split]
pop_X = pop_raw.loc[:, pop_columns].to_numpy(dtype=float)
assert np.isfinite(pop_X).all() and ((pop_X >= 0) & (pop_X <= 2)).all()
for mask in pop_masks.values():
    frequency = pop_X[mask].mean(axis=0) / 2
    assert (np.minimum(frequency, 1-frequency) >= 0.05-1e-12).all()
pop_order = np.argsort(pop_variants.position.to_numpy(), kind="stable")
pop_variants = pop_variants.iloc[pop_order].reset_index(drop=True)
pop_X = pop_X[:, pop_order]
pop_positions = pop_variants.position.to_numpy()
pop_Y = expression_by_gene.loc[:, pop_raw.IID].astype(float)
assert np.isfinite(pop_Y.to_numpy()).all()
pop_genes = genes.loc[genes.chrom.eq(22)].set_index("gene_id")
print(f"Chromosome 22: {pop_masks['EUR'].sum()} EUR, {pop_masks['YRI'].sum()} YRI; "
      f"{len(pop_variants):,} shared common SNPs; {len(pop_genes)} measured annotated genes.")

pop_pca_prefix = POP_OUT / "work/pca_markers"
run_plink(["--bfile", POP_PREFIX, "--indep-pairwise", "200", "50", "0.2", "--out", pop_pca_prefix])
pop_marker_ids = set(Path(str(pop_pca_prefix) + ".prune.in").read_text().splitlines())
pop_marker_mask = pop_variants.snp.isin(pop_marker_ids).to_numpy()
pop_context = relationship_context(pop_X, pop_positions, pop_marker_mask)
pop_variants.loc[pop_marker_mask, ["snp", "position"]].to_csv(POP_OUT / "pca_markers.tsv", sep="\t", index=False)
print(f"Population PCA marker set: {int(pop_marker_mask.sum()):,} LD-pruned SNPs; each gene excludes its cis window.")


Chromosome 22: 358 EUR, 87 YRI; 65,266 shared common SNPs; 633 measured annotated genes.
Population PCA marker set: 3,547 LD-pruned SNPs; each gene excludes its cis window.


In [7]:
pop_pair_file = POP_OUT / "work/all_interaction_pairs.tsv.gz"
pop_fields = ["chrom", "gene_id", "symbol", "snp", "position", "counted_allele", "other_allele",
              "n_eur", "n_yri", "beta_eur", "beta_yri", "beta_difference", "se_difference_hc3", "p_interaction"]
pop_gene_summary = []
pop_tests = pop_checks = 0
with gzip.open(pop_pair_file, "wt") as handle:
    handle.write("\t".join(pop_fields) + "\n")
    for gene_number, (gene_id, gene) in enumerate(pop_genes.iterrows()):
        y = pop_Y.loc[gene_id].to_numpy(dtype=float)
        left = np.searchsorted(pop_positions, int(gene.window_start), side="left")
        right = np.searchsorted(pop_positions, int(gene.window_end), side="right")
        indices = np.arange(left, right)
        if not len(indices) or any(np.ptp(y[mask]) == 0 for mask in pop_masks.values()):
            pop_gene_summary.append({"gene_id": gene_id, "tests": 0, "lead_p_interaction": np.nan})
            continue
        x = pop_X[:, indices]
        pop_covariates, pca_markers_used = population_covariates(
            pop_context, int(gene.window_start), int(gene.window_end),
            pop_masks["YRI"], N_PCS)
        beta_e, beta_y, difference, se_difference, p_interaction, valid = interaction_hc3(
            y, x, pop_masks["YRI"], pop_covariates)
        indices = indices[valid]
        if not len(indices):
            pop_gene_summary.append({"gene_id": gene_id, "tests": 0, "lead_p_interaction": np.nan})
            continue
        if pop_checks == 0 or gene_number % 100 == 0:
            k = int(np.argmin(np.abs(p_interaction-0.1)))
            check_interaction(y, pop_X[:, indices[k]], pop_masks["YRI"], pop_covariates,
                              difference[k], se_difference[k], p_interaction[k])
            pop_checks += 1
        variants = pop_variants.iloc[indices]
        result = pd.DataFrame({"chrom": 22, "gene_id": gene_id, "symbol": gene.symbol,
            "snp": variants.snp.to_numpy(), "position": variants.position.to_numpy(),
            "counted_allele": variants.counted_allele.to_numpy(),
            "other_allele": np.where(variants.counted_allele.eq(variants.a1), variants.a2, variants.a1),
            "n_eur": int(pop_masks["EUR"].sum()), "n_yri": int(pop_masks["YRI"].sum()),
            "beta_eur": beta_e, "beta_yri": beta_y, "beta_difference": difference,
            "se_difference_hc3": se_difference, "p_interaction": p_interaction})
        result.to_csv(handle, sep="\t", index=False, header=False)
        pop_tests += len(result)
        pop_gene_summary.append({"gene_id": gene_id, "tests": len(result), "lead_p_interaction": p_interaction.min(), "pca_markers_used": pca_markers_used})
assert pop_tests > 0
pop_gene_summary = pd.DataFrame(pop_gene_summary)
assert int(pop_gene_summary.tests.sum()) == pop_tests
pop_gene_summary.to_csv(POP_OUT / "gene_interaction_summary.tsv", sep="\t", index=False)
print(f"{pop_tests:,} interaction tests on chromosome 22; {pop_checks} direct HC3 checks passed.")


1,137,206 interaction tests on chromosome 22; 7 direct HC3 checks passed.


In [8]:
pop_threshold = ALPHA / pop_tests
pop_significant_file = POP_OUT / "significant_interaction_pairs.tsv.gz"
pop_significant_count = pop_rows_read = 0
pop_significant_genes = set()
pop_top = []
with gzip.open(pop_significant_file, "wt") as handle:
    handle.write("\t".join(pop_fields + ["p_bonferroni_interaction"]) + "\n")
    for chunk in pd.read_csv(pop_pair_file, sep="\t", chunksize=100_000):
        pop_rows_read += len(chunk)
        assert chunk.p_interaction.between(0, 1).all()
        significant = chunk.loc[chunk.p_interaction.le(pop_threshold)].copy()
        significant["p_bonferroni_interaction"] = np.minimum(significant.p_interaction*pop_tests, 1)
        significant.to_csv(handle, sep="\t", index=False, header=False)
        pop_significant_count += len(significant)
        pop_significant_genes.update(significant.gene_id)
        if not significant.empty:
            pop_top.append(significant.nsmallest(5, "p_interaction"))
assert pop_rows_read == pop_tests
pop_answer = {"chromosome": 22, "n_eur": 358, "n_yri": 87,
              "tested_pairs": pop_tests, "significant_interaction_pairs": pop_significant_count,
              "genes_with_significant_interaction": len(pop_significant_genes),
              "bonferroni_threshold": pop_threshold, "alpha": ALPHA,
              "model": "SNP + population + SNP:population + 5 genotype PCs; HC3 normal Wald test",
              "pca_source": "LD-pruned chromosome 22 markers excluding each gene cis window",
              "maf_min_each_population": 0.05, "genotype_pcs": N_PCS, "cis_flank_bp": CIS_FLANK_BP}
(POP_OUT / "population_answer.json").write_text(json.dumps(pop_answer, indent=2) + "\n")
print(f"ANSWER: {pop_significant_count:,} PCA-adjusted population-dependent SNP–gene effects on chromosome 22.")
print(f"Genes involved: {len(pop_significant_genes)}; tests: {pop_tests:,}; threshold: {pop_threshold:.6g}")
if pop_top:
    display(pd.concat(pop_top).nsmallest(5, "p_interaction"))
else:
    print("No significant interactions.")
print("Results:", POP_OUT)


ANSWER: 341 PCA-adjusted population-dependent SNP–gene effects on chromosome 22.
Genes involved: 21; tests: 1,137,206; threshold: 4.39674e-08
Results: results/population_chr22_pca


,chrom,gene_id,symbol,snp,position,counted_allele,other_allele,n_eur,n_yri,beta_eur,beta_yri,beta_difference,se_difference_hc3,p_interaction,p_bonferroni_interaction
1032005,22,ENSG00000075234,TTC38,22:46704243:C:T,46704243,C,T,358,87,1.040567,-4.493986,-5.534553,0.424446,7.296022e-39,8.297080e-33
1031998,22,ENSG00000075234,TTC38,22:46701991:A:G,46701991,A,G,358,87,1.051178,-4.494357,-5.545535,0.426006,9.730786e-39,1.106591e-32
1031999,22,ENSG00000075234,TTC38,22:46702449:T:C,46702449,T,C,358,87,1.051178,-4.494357,-5.545535,0.426006,9.730786e-39,1.106591e-32
1032001,22,ENSG00000075234,TTC38,22:46703664:T:C,46703664,T,C,358,87,1.051178,-4.494357,-5.545535,0.426006,9.730786e-39,1.106591e-32
1032002,22,ENSG00000075234,TTC38,22:46703776:C:T,46703776,C,T,358,87,1.051178,-4.494357,-5.545535,0.426006,9.730786e-39,1.106591e-32


# Results

In [9]:
scope = "22" if ONLY_22 else "1–22"
result_rows = [
    {"question": "Significant SNP–gene pairs", "chromosomes": scope, "chromosome_Bonferroni": chrom_total, "scan_Bonferroni": scan_total},
    {"question": "eGenes", "chromosomes": scope, "chromosome_Bonferroni": int(egenes.eGenes_chromosome.sum()), "scan_Bonferroni": int(egenes.eGenes_scan.sum())},
    {"question": "Population-dependent effects", "chromosomes": "22", "chromosome_Bonferroni": pop_significant_count, "scan_Bonferroni": pop_significant_count}]
print("Results")
display(pd.DataFrame(result_rows))
print("Pairs can share a gene or reflect the same underlying signal.")
print("PCA adjusts associations for measured structure; these counts do not prove causality or fewer false positives.")


Results
Pairs can share a gene or reflect the same underlying signal.
PCA adjusts associations for measured structure; these counts do not prove causality or fewer false positives.


,question,chromosomes,chromosome_Bonferroni,scan_Bonferroni
0,Significant SNP–gene pairs,22,1302,1302
1,eGenes,22,74,74
2,Population-dependent effects,22,341,341
